# Persona 2 - Recuperacion Clasica (TF-IDF y BM25)
### Proyecto Bimestre 1 - Recuperacion de Informacion

## 1. Importaciones y configuracion

In [3]:
import pandas as pd
import numpy as np
import nltk

In [4]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict, Counter

In [5]:
import string
import math

In [6]:
# Descarga de recursos necesarios de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 2. Carga del corpus
Se utiliza el dataset Reuters-21578 (ModApte split).  
Se eliminan las filas que no tienen texto.

In [7]:
# Cargar el dataset
df = pd.read_csv('../../ModApte_test.csv')

# Eliminar documentos sin texto
df = df.dropna(subset=['text']).reset_index(drop=True)

print(f'Documentos cargados: {len(df)}')
print(f'Columnas disponibles: {df.columns.tolist()}')

Documentos cargados: 3023
Columnas disponibles: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']


## 3. Preprocesamiento
Se aplica a cada documento:
- Conversion a minusculas
- Tokenizacion
- Eliminacion de signos de puntuacion
- Eliminacion de stopwords en ingles
- Eliminacion de tokens cortos (menor a 3 caracteres)

In [8]:
# Cargar stopwords en ingles
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Convertir a minusculas
    text = text.lower()

    # Tokenizar el texto
    tokens = word_tokenize(text)

    # Limpiar tokens: eliminar puntuacion, stopwords y tokens cortos
    tokens = [
        t.replace('.', '').replace('-', ' ').strip()
        for t in tokens
        if t not in string.punctuation
        and t not in stop_words
        and len(t) > 2
    ]

    # Eliminar tokens vacios que quedaron tras el reemplazo
    tokens = [t for t in tokens if t]

    return tokens

# Verificar con el primer documento
print('Ejemplo de tokens del primer documento:')
print(preprocess(df['text'].iloc[0])[:20])

Ejemplo de tokens del primer documento:
['mounting', 'trade', 'friction', 'us', 'japan', 'raised', 'fears', 'among', 'many', 'asia', 'exporting', 'nations', 'row', 'could', 'inflict', 'far reaching', 'economic', 'damage', 'businessmen', 'officials']


## 4. Indice invertido y estadisticas
Se construyen estructuras para recuperacion clasica:
- indice invertido (termino -> {doc_id: frecuencia})
- frecuencia de documento (df)
- longitud de documento y promedio

In [9]:
# Preparar columnas de identificación
if 'new_id' in df.columns:
    # Convertir new_id a numérico; los no numéricos se convierten a NaN
    doc_ids = pd.to_numeric(df['new_id'], errors='coerce')
    # Llenar NaN con el índice de la fila (0,1,2,...) convertido a Serie
    df['doc_id'] = doc_ids.fillna(pd.Series(df.index, index=df.index)).astype(int)
else:
    df['doc_id'] = df.index.astype(int)

In [10]:
# Tokenizar todos los documentos (asegúrate de tener la función preprocess definida)
df['tokens'] = df['text'].apply(preprocess)

In [11]:
# Estructuras principales
inverted_index = defaultdict(dict)
doc_term_freqs = {}
doc_len = {}

for _, row in df.iterrows():
    doc_id = int(row['doc_id'])
    tokens = row['tokens']

    term_counts = Counter(tokens)
    doc_term_freqs[doc_id] = term_counts
    doc_len[doc_id] = sum(term_counts.values())

    for term, freq in term_counts.items():
        inverted_index[term][doc_id] = freq

doc_freq = {term: len(postings) for term, postings in inverted_index.items()}
N = len(doc_term_freqs)
avg_doc_len = sum(doc_len.values()) / N

print(f'Documentos indexados: {N}')
print(f'Términos en vocabulario: {len(inverted_index)}')

Documentos indexados: 3023
Términos en vocabulario: 23141


## 5. Modelos clasicos de recuperacion
Se implementan dos modelos:
- TF-IDF + similitud coseno
- BM25

In [12]:
# Calcular IDF para cada término
# Fórmula: idf = log(N / df)
idf = {}
for term, df in doc_freq.items():
    idf[term] = math.log(N / df)

In [14]:
# Calcular pesos TF-IDF
# Estructura: doc_tfidf[doc_id][termino] = tf * idf
doc_tfidf = {}
for doc_id, term_counts in doc_term_freqs.items():
    weights = {}
    for term, tf in term_counts.items():
        weights[term] = tf * idf.get(term, 0)
    doc_tfidf[doc_id] = weights

In [15]:
# Calcular norma (longitud) de cada documento
# Norma = sqrt(sum(peso^2))
doc_norms = {}
for doc_id, weights in doc_tfidf.items():
    suma = sum(w * w for w in weights.values())
    doc_norms[doc_id] = math.sqrt(suma)

In [16]:
# Procesar consulta: obtener pesos TF-IDF y su norma
def procesar_consulta(query):
    tokens = preprocess(query)            # mismo preprocesamiento
    tf_q = Counter(tokens)                # frecuencias en consulta
    pesos_q = {}
    for term, freq in tf_q.items():
        if term in idf:                   # solo términos conocidos
            pesos_q[term] = freq * idf[term]
    norma_q = math.sqrt(sum(w*w for w in pesos_q.values()))
    return pesos_q, norma_q

In [17]:
# Similitud coseno usando índice invertido
def similitud_coseno(pesos_q, norma_q):
    puntajes = {}
    for term, wq in pesos_q.items():
        if term not in inverted_index:
            continue
        for doc_id, tf_d in inverted_index[term].items():
            wd = tf_d * idf[term]           # peso tf-idf del documento
            puntajes[doc_id] = puntajes.get(doc_id, 0) + wq * wd
    # Normalizar por las normas
    for doc_id in puntajes:
        if doc_norms[doc_id] > 0 and norma_q > 0:
            puntajes[doc_id] /= (doc_norms[doc_id] * norma_q)
        else:
            puntajes[doc_id] = 0
    return puntajes

In [18]:
# Función de búsqueda principal
def buscar_tfidf(consulta, top_k=10):
    pesos_q, norma_q = procesar_consulta(consulta)
    if norma_q == 0:
        return []
    puntajes = similitud_coseno(pesos_q, norma_q)
    ranking = sorted(puntajes.items(), key=lambda x: x[1], reverse=True)
    return ranking[:top_k]

In [29]:
# Ejemplo de uso
resultados = buscar_tfidf("car insurance", 10)
for doc_id, score in resultados:
    print(f"Documento {doc_id}: {score:.4f}")

Documento 1355: 0.2956
Documento 1416: 0.2457
Documento 596: 0.2302
Documento 499: 0.2199
Documento 2139: 0.2114
Documento 220: 0.1542
Documento 560: 0.1510
Documento 1438: 0.1480
Documento 1273: 0.1428
Documento 1274: 0.1346


In [35]:
# Parámetros del modelo BM25(valores típicos)
k1 = 1.5      # Controla saturación de tf en documento
b = 0.75      # Controla normalización por longitud
k3 = 1.5      # Controla saturación de tf en consulta (para consultas largas)

In [36]:
# Calcular IDF versión robusta (con +0.5)
idf_bm25 = {}
for term, df in doc_freq.items():
    idf_bm25[term] = math.log((N - df + 0.5) / (df + 0.5))

In [34]:
# Función para obtener frecuencias de términos en la consulta
def get_query_tf(query):
    tokens = preprocess(query)           # Mismo preprocesamiento que documentos
    return Counter(tokens)               # tf_q: frecuencia de cada término en consulta

In [37]:
# Función para calcular score BM25 de un documento dado término (con k3)
def bm25_term_score(term, doc_id, tf_d, doc_len, avg_dl, tf_q, k3):
    idf_val = idf_bm25.get(term, 0)
    if idf_val <= 0:
        return 0
    # Normalización por longitud del documento
    B = (1 - b) + b * (doc_len / avg_dl)
    # Factor por frecuencia en documento (k1)
    factor_d = ((k1 + 1) * tf_d) / (k1 * B + tf_d)
    # Factor por frecuencia en consulta (k3)
    if k3 > 0:
        factor_q = ((k3 + 1) * tf_q) / (k3 + tf_q)
    else:
        factor_q = 1.0
    return idf_val * factor_d * factor_q

In [39]:
# Búsqueda BM25
def buscar_bm25(consulta, top_k=10):
    tf_q = get_query_tf(consulta)           # Diccionario {término: frecuencia en consulta}
    scores = {}
    for term, q_freq in tf_q.items():
        if term not in inverted_index:
            continue
        postings = inverted_index[term]    # {doc_id: tf_d}
        for doc_id, tf_d in postings.items():
            score_term = bm25_term_score(term, doc_id, tf_d,
                                         doc_len[doc_id], avg_doc_len,
                                         q_freq, k3)
            if score_term > 0:
                scores[doc_id] = scores.get(doc_id, 0.0) + score_term
    # Ordenar por puntuación descendente
    ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranking[:top_k]

In [41]:
# Ejemplo de uso
resultados = buscar_bm25("car insurance", top_k=10)
for doc_id, score in resultados:
    print(f"Documento {doc_id}: BM25 score = {score:.4f}")

Documento 499: BM25 score = 8.5982
Documento 383: BM25 score = 8.4835
Documento 2139: BM25 score = 8.2090
Documento 1355: BM25 score = 7.8757
Documento 1273: BM25 score = 7.7515
Documento 1416: BM25 score = 7.5891
Documento 1438: BM25 score = 7.4494
Documento 596: BM25 score = 7.4409
Documento 510: BM25 score = 7.1777
Documento 220: BM25 score = 7.1092


## 6. Consulta de texto libre y ranking
Se ejecuta una consulta y se muestran los resultados ordenados por relevancia.

In [14]:
def show_results(ranked_docs, model_name='Modelo', top_n=5):
    print(f'\n[{model_name}] Top {min(top_n, len(ranked_docs))} resultados')

    if not ranked_docs:
        print('No se encontraron resultados.')
        return

    for rank, (doc_id, score) in enumerate(ranked_docs[:top_n], start=1):
        row = df[df['doc_id'] == doc_id].iloc[0]
        title = str(row['title']) if 'title' in df.columns and pd.notna(row['title']) else ''
        text = str(row['text']).replace('\n', ' ')
        snippet = text[:140] + ('...' if len(text) > 140 else '')

        print(f'{rank}. doc_id={doc_id} | score={score:.5f}')
        if title:
            print(f'   title: {title}')
        print(f'   text : {snippet}')

query = 'trade japan market'

tfidf_results = search_tfidf_cosine(query, top_k=10)
bm25_results = search_bm25(query, top_k=10)

print(f'Consulta: {query}')
show_results(tfidf_results, model_name='TF-IDF + Coseno', top_n=5)
show_results(bm25_results, model_name='BM25', top_n=5)

Consulta: trade japan market

[TF-IDF + Coseno] Top 5 resultados
1. doc_id=568 | score=0.26142
   title: JAPAN BUYS 5,000 TONNES CANADIAN RAPESEED
  Reuterources said.at an undisclosed price for May shipment,
2. doc_id=319 | score=0.24521
   title: JAPAN MINISTRY HAS NO COMMENT ON RICE TALKS REPORT
 on its closed rice market in the...pan had agreed to hold talks
3. doc_id=889 | score=0.23076
   title: TRADE ISSUES STRAINING EC'S PATIENCE WITH JAPAN
 they believe has repeatedly promised major in...th Japan which
4. doc_id=1203 | score=0.21939
   title: U.S. URGES JAPAN TO OPEN FARM MARKET FURTHER
 Washington cut its trade deficit and ease ...er to help
5. doc_id=1222 | score=0.21939
   title: U.S. URGES JAPAN TO OPEN FARM MARKET FURTHER
 Washington cut its trade deficit and ease ...er to help

[BM25] Top 5 resultados
1. doc_id=319 | score=23.29732
   title: JAPAN MINISTRY HAS NO COMMENT ON RICE TALKS REPORT
 on its closed rice market in the...pan had agreed to hold talks
2. doc_id=1203 